# 데이터 로드

In [ ]:
# manipulated
import gdown

url = "https://drive.google.com/uc?id=1Dy8OjqVSQ4gvZiF70LruptrjccgIm9J3"
gdown.download(url,'/content/manipulated.zip',quiet=False)

In [ ]:
!unzip /content/manipulated.zip -d /content

In [ ]:
# original

url = "https://drive.google.com/uc?id=1F3NvAGSz-6kDY5SPZjZy6wZYE89iemD4"
gdown.download(url,'/content/original.zip',quiet=False)

In [ ]:
!unzip /content/original.zip -d /content

In [ ]:
!mv /content/DFD_manipulated_sequences /content/manipulated

In [ ]:
!mv "/content/DFD_original sequences" "/content/original"

## 샘플 수 확인

In [ ]:
import os
import glob

root_dir = '/content'  # 본인 폴더로 수정

for subdir in ['original', 'manipulated']:
    sub_path = os.path.join(root_dir, subdir)
    count = len(glob.glob(os.path.join(sub_path, '*.jpg')))
    print(f"{subdir} 폴더의 jpg 파일 개수: {count}개")


# 모델 세팅(로스, 데이터셋 등)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.models.video import swin3d_t, Swin3D_T_Weights
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, matthews_corrcoef
import matplotlib.pyplot as plt
import albumentations as A
import cv2
from tqdm import tqdm


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        BCE = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-BCE)
        F_loss = self.alpha * (1 - pt) ** self.gamma * BCE
        return F_loss.mean()


In [ ]:
class DeepfakeVideoDataset(Dataset):
    def __init__(self, root_dir, frame_num=72, size=224):
        self.data = []
        self.labels = []
        self.frame_num = frame_num
        self.size = size
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        for label, subdir in enumerate(['original', 'manipulated']):
            sub_path = os.path.join(root_dir, subdir)
            video_dict = {}
            for img_path in glob.glob(os.path.join(sub_path, "*.jpg")):
                fname = os.path.basename(img_path)
                main, ext = os.path.splitext(fname)
                parts = main.split('_')
                if len(parts) < 3:
                    continue  # 불량 파일 무시
                # robust parsing (뒤에서부터)
                try:
                    face_idx = int(parts[-1])
                    fnum = int(parts[-2])
                except ValueError:
                    continue
                video_id = '_'.join(parts[:-2])
                # 비디오별/인물별 프레임별 구조
                if video_id not in video_dict:
                    video_dict[video_id] = {}
                if face_idx not in video_dict[video_id]:
                    video_dict[video_id][face_idx] = {}
                video_dict[video_id][face_idx][fnum] = img_path

            # 인물별 시퀀스 완성 (모든 프레임이 있는 인물만)
            for video_id, people_dict in video_dict.items():
                for face_idx, frame_dict in people_dict.items():
                    if len(frame_dict) == self.frame_num:
                        frames = [frame_dict[fidx] for fidx in range(self.frame_num)]
                        self.data.append((video_id, face_idx, frames))
                        self.labels.append(label)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        video_id, face_idx, frames = self.data[idx]
        label = self.labels[idx]
        imgs = []
        for img_path in frames:
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.size, self.size))
            img = self.transform(img)
            imgs.append(img)
        # (T, 3, 224, 224) -> (3, T, 224, 224)
        video_tensor = torch.stack(imgs, dim=0).permute(1, 0, 2, 3)
        return video_tensor, label, video_id, face_idx


In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset


# 데이터셋 경로
root_dir = '/content'   # 예시: /content/faces/original, /content/faces/manipulated

dataset = DeepfakeVideoDataset(root_dir, frame_num=72, size=224)

# 데이터셋의 라벨 리스트를 사용해서 인덱스 분할
indices = np.arange(len(dataset))
labels = [dataset.labels[i] for i in range(len(dataset))]

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

train_set = Subset(dataset, train_idx)
val_set = Subset(dataset, val_idx)

train_loader = DataLoader(train_set, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=4, shuffle=False, num_workers=2)


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

weights = Swin3D_T_Weights.DEFAULT
model = swin3d_t(weights=weights)
model.head = nn.Linear(model.head.in_features, 2)  # 이진 분류
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = FocalLoss(alpha=0.25, gamma=2)  # 클래스 비율 맞게 alpha 조정 가능


In [ ]:
from collections import defaultdict

In [ ]:
num_epochs = 30
patience = 10
best_auc = 0
pat = 0
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[],
           'train_f1':[], 'val_f1':[], 'train_auc':[], 'val_auc':[],
           'train_mcc':[], 'val_mcc':[],
           'video_acc':[], 'video_f1':[], 'video_auc':[], 'video_mcc':[]   # <= 추가!
}

for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    train_loss, preds, targets = [], [], []
    for video_tensor, label, video_id, face_idx in tqdm(train_loader, desc=f'Epoch {epoch+1} [Train]'):
        video_tensor = video_tensor.to(device)  # (batch, 3, 72, 224, 224)
        label = label.to(device)
        optimizer.zero_grad()
        out = model(video_tensor)  # (batch, 2)
        loss = criterion(out, label)
        loss.backward()
        optimizer.step()
        train_loss.append(loss.item())
        prob = out.softmax(-1)[:, 1].detach().cpu().numpy()
        pred_label = (prob > 0.5).astype(int)
        preds += list(pred_label)
        targets += list(label.cpu().numpy())
    train_loss = np.mean(train_loss)
    train_acc = accuracy_score(targets, preds)
    train_f1 = f1_score(targets, preds)
    train_auc = roc_auc_score(targets, preds)
    train_mcc = matthews_corrcoef(targets, preds)

    # --- Validation ---
    model.eval()
    val_loss, preds, targets = [], [], []
    val_video_ids, val_face_idxs = [], []
    with torch.no_grad():
        for video_tensor, label, video_id, face_idx in tqdm(val_loader, desc=f'Epoch {epoch+1} [Val]'):
            video_tensor = video_tensor.to(device)
            label = label.to(device)
            out = model(video_tensor)
            loss = criterion(out, label)
            val_loss.append(loss.item())
            prob = out.softmax(-1)[:, 1].cpu().numpy()
            pred_label = (prob > 0.5).astype(int)
            preds += list(pred_label)
            targets += list(label.cpu().numpy())
            # video_id, face_idx는 batch 단위일 수 있으니 list로 변환
            if isinstance(video_id, (list, tuple)):
                val_video_ids += list(video_id)
                val_face_idxs += list(face_idx)
            else:
                val_video_ids.append(video_id)
                val_face_idxs.append(face_idx)
    val_loss = np.mean(val_loss)
    val_acc = accuracy_score(targets, preds)
    val_f1 = f1_score(targets, preds)
    val_auc = roc_auc_score(targets, preds)
    val_mcc = matthews_corrcoef(targets, preds)

    # --- 비디오 level 평가 ---
    video_pred_dict = defaultdict(list)
    video_target_dict = {}

    for vid, pred, target in zip(val_video_ids, preds, targets):
        video_pred_dict[vid].append(pred)
        video_target_dict[vid] = target  # 동일 비디오 내 label은 동일

    video_preds = []
    video_targets = []
    for vid in video_pred_dict:
        vote = np.mean(video_pred_dict[vid])
        video_preds.append(int(vote > 0.5))
        video_targets.append(video_target_dict[vid])

    video_acc = accuracy_score(video_targets, video_preds)
    video_f1 = f1_score(video_targets, video_preds)
    video_auc = roc_auc_score(video_targets, video_preds)
    video_mcc = matthews_corrcoef(video_targets, video_preds)

    # 기록
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    history['train_auc'].append(train_auc)
    history['val_auc'].append(val_auc)
    history['train_mcc'].append(train_mcc)
    history['val_mcc'].append(val_mcc)
    # --- Video-level 기록 추가 ---
    history['video_acc'].append(video_acc)
    history['video_f1'].append(video_f1)
    history['video_auc'].append(video_auc)
    history['video_mcc'].append(video_mcc)

    print(f"[{epoch+1}] train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, "
          f"val_f1={val_f1:.4f}, val_auc={val_auc:.4f}, val_mcc={val_mcc:.4f}")
    print(f"     (Video-level) acc={video_acc:.4f}, f1={video_f1:.4f}, auc={video_auc:.4f}, mcc={video_mcc:.4f}")

    # Early Stopping & Checkpoint
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), 'best_videoswin.pth')
        pat = 0
    else:
        pat += 1
    if pat >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

# 시각화 (그래프 및 혼동행렬)

## 그래프

In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", font_scale=1.2)
epochs = len(history['train_loss'])
x = np.arange(1, epochs + 1)


In [ ]:
# --- 1. Loss (단일 plot) ---
plt.figure(figsize=(7, 5))
plt.plot(x, history['train_loss'], label='Train', color='#2874A6', linewidth=2.2, marker='o', markersize=6)
plt.plot(x, history['val_loss'], label='Val', color='#CB4335', linewidth=2.2, marker='s', markersize=6)
best_idx = np.argmin(history['val_loss'])
plt.scatter(x[best_idx], history['val_loss'][best_idx],
            color='#CB4335', s=120, edgecolor='black', label='Best Val', zorder=5)
plt.title('Loss per Epoch', fontsize=17, weight='bold')
plt.xlabel('Epoch', fontsize=13)
plt.ylabel('Loss', fontsize=13)
plt.ylim(bottom=0)
plt.grid(alpha=0.25)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. 네 지표(2행 2열) ---
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
titles = ['Accuracy', 'F1-score', 'AUC', 'MCC']
metrics = [
    ('train_acc', 'val_acc'),
    ('train_f1', 'val_f1'),
    ('train_auc', 'val_auc'),
    ('train_mcc', 'val_mcc'),
]
train_color = '#2874A6'
val_color = '#CB4335'

for idx, ax in enumerate(axes.flat):
    tr, va = metrics[idx]
    ax.plot(x, history[tr], label='Train', color=train_color, linewidth=2.2, marker='o', markersize=6)
    ax.plot(x, history[va], label='Val', color=val_color, linewidth=2.2, marker='s', markersize=6)
    ax.set_title(titles[idx], fontsize=17, weight='bold')
    ax.set_xlabel('Epoch', fontsize=13)
    ax.set_ylabel(titles[idx], fontsize=13)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=12, loc='best')
    # Best epoch marker for val
    best_idx = np.argmax(history[va])
    ax.scatter(x[best_idx], history[va][best_idx],
               color=val_color, s=120, edgecolor='black',
               label='Best Val', zorder=5)
plt.suptitle("Training vs. Validation Metrics (except Loss)", fontsize=21, weight='bold', color='#1a1a1a', y=1.02)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
# --------------- 비디오 레벨(지표마다 색상 다름) ---------------
sns.set_theme(style="whitegrid", font_scale=1.25)
palette = sns.color_palette("husl", 4)  # 지표마다 다양한 컬러, 세련된 팔레트

metric_keys = ['video_acc', 'video_f1', 'video_auc', 'video_mcc']
metric_names = ['Accuracy', 'F1-score', 'AUC', 'MCC']
markers = ['o', 's', '^', 'D']

plt.figure(figsize=(10, 6))

for i, (key, name, marker) in enumerate(zip(metric_keys, metric_names, markers)):
    plt.plot(x, history[key], label=name, color=palette[i], marker=marker, linewidth=2.5, markersize=8)
    # Best marker & 텍스트
    best_idx = np.argmax(history[key])
    plt.scatter(x[best_idx], history[key][best_idx], color=palette[i], s=170, edgecolor='black', zorder=5, marker=marker, linewidth=1.7)
    plt.text(x[best_idx], history[key][best_idx]+0.025,
             f"Best: {history[key][best_idx]:.3f}\n(Epoch {x[best_idx]})",
             color=palette[i], fontsize=12, fontweight='bold',
             ha='center', va='bottom', bbox=dict(boxstyle="round,pad=0.17", fc="white", ec=palette[i], lw=1.3, alpha=0.9))

plt.title("Validation Metrics per Epoch (Video Level)", fontsize=21, weight='bold', color="#212121")
plt.xlabel("Epoch", fontsize=15)
plt.ylabel("Metric Value", fontsize=15)
plt.ylim(0, 1.07)
plt.xticks(x)
plt.grid(alpha=0.22)
plt.legend(fontsize=13, loc='lower right', frameon=True, shadow=True, facecolor='white')
plt.tight_layout()
plt.show()

## 혼동행렬

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# 예측 (개별 인물 시퀀스 기준, threshold 0.5)
val_pred_label = [1 if p > 0.5 else 0 for p in preds]
cm = confusion_matrix(targets, val_pred_label)
labels = ['Original', 'Manipulated']

plt.figure(figsize=(5.5, 4.8))
sns.set_theme(style="white", font_scale=1.15)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues", values_format='d', ax=plt.gca(), colorbar=True)
plt.title("Confusion Matrix\n(Individual Sequence Level)", fontsize=13, weight='bold')
plt.xlabel("Predicted Label", fontsize=13)
plt.ylabel("True Label", fontsize=13)
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# 예측 (비디오 level, majority voting 등으로 집계 후)
cm_video = confusion_matrix(video_targets, video_preds)
plt.figure(figsize=(5.5, 4.8))
sns.set_theme(style="white", font_scale=1.15)
disp_video = ConfusionMatrixDisplay(confusion_matrix=cm_video, display_labels=labels)
disp_video.plot(cmap="BuGn", values_format='d', ax=plt.gca(), colorbar=True)
plt.title("Confusion Matrix\n(Video Level)", fontsize=13, weight='bold')
plt.xlabel("Predicted Label", fontsize=13)
plt.ylabel("True Label", fontsize=13)
plt.grid(False)
plt.tight_layout()
plt.show()

## 오분류 샘플

In [ ]:
# --------------- 개별 시퀀스 레벨 오분류 샘플 ---------------
val_pred_label = [1 if p > 0.5 else 0 for p in preds]  # or np.array
mis_idx = [i for i, (t, p) in enumerate(zip(targets, val_pred_label)) if t != p]

# (video_id, face_idx, 실제 정답, 예측값, 예측확률) 튜플로 정리
misclassified = [(val_video_ids[i], val_face_idxs[i], targets[i], val_pred_label[i], preds[i]) for i in mis_idx]

# 1종 오류(원본을 조작으로 오분류)
type1 = [row for row in misclassified if row[2]==0 and row[3]==1]
# 2종 오류(조작을 원본으로 오분류)
type2 = [row for row in misclassified if row[2]==1 and row[3]==0]

print(f"1종 오류(원본->조작): {len(type1)}개")
print(f"2종 오류(조작->원본): {len(type2)}개")
print()
# 예측확률 내림차순
type1_sorted = sorted(type1, key=lambda x: -x[4])
type2_sorted = sorted(type2, key=lambda x: -x[4])

# 일부 예시 출력
for v_id, fidx, label, pred, prob in type1_sorted[:10]:
    print(f"[1종] {v_id:20s} {fidx:2d}  정답: {label}  예측: {pred} (manip prob: {prob:.2f})")
print("")
for v_id, fidx, label, pred, prob in type2_sorted[:10]:
    print(f"[2종] {v_id:20s} {fidx:2d}  정답: {label}  예측: {pred} (manip prob: {prob:.2f})")

In [ ]:
# --------------- 개별 시퀀스 레벨 오분류 샘플 ---------------
video_ids_unique = list(video_pred_dict.keys())

mis_idx_video = [i for i, (t, p) in enumerate(zip(video_targets, video_preds)) if t != p]

# 각 비디오별 예측 확률(평균/최대 등), 여기서는 평균 사용
video_probs = [np.mean(video_pred_dict[vid]) for vid in video_ids_unique]

# 오분류 샘플 (비디오 ID, 실제 정답, 예측값, 예측확률)
misclassified_video = [(video_ids_unique[i], video_targets[i], video_preds[i], video_probs[i]) for i in mis_idx_video]

# 1종 오류(원본->조작)
type1_v = [row for row in misclassified_video if row[1]==0 and row[2]==1]
# 2종 오류(조작->원본)
type2_v = [row for row in misclassified_video if row[1]==1 and row[2]==0]

print(f"1종 오류(원본->조작): {len(type1_v)}개")
print(f"2종 오류(조작->원본): {len(type2_v)}개\n")

# 예측확률 내림차순 정렬
type1_v_sorted = sorted(type1_v, key=lambda x: -x[3])
type2_v_sorted = sorted(type2_v, key=lambda x: -x[3])

# 상위 10개 출력
for v_id, label, pred, prob in type1_v_sorted[:10]:
    print(f"[1종] {v_id:20s}  정답: {label}  예측: {pred} (mean prob: {prob:.2f})")
print("")
for v_id, label, pred, prob in type2_v_sorted[:10]:
    print(f"[2종] {v_id:20s}  정답: {label}  예측: {pred} (mean prob: {prob:.2f})")

In [ ]:
for v_id, label, pred, prob in misclassified_video[:5]:
    print(f"\n비디오ID: {v_id}, 실제: {label}, 예측: {pred}, 비디오 예측확률(평균): {prob:.2f}")
    seq_preds = video_pred_dict[v_id]  # 시퀀스별 예측(0/1) 리스트
    print("프레임별 예측:", seq_preds)
    print("프레임별 1 확률 평균:", np.mean(seq_preds))
    # 만약 확률 저장한 경우(0~1), np.array로
    # 프레임 인덱스/이미지와 함께 시각화 가능

## 히트맵

In [ ]:
import torch
import torch.nn.functional as F